# Módulo 09 · Lista de Exercícios

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Este módulo tira você do caminho crítico. Ao final, o deploy não depende de você estar disponível — e o sistema avisa antes do cliente.

## Como usar

| | |
|---|---|
| 🟢 **Aquecimento** | 1–10 · uma ideia por exercício |
| 🟡 **Construção** | 11–26 · combinam conceitos |
| 🔴 **Integração** | 27–40 · perto de produção |
| 🏗️ **Projeto** | O Atlas no ar, sozinho |

**Regras de casa:**

1. Os exercícios de servidor rodam aqui. Os de VPS exigem um servidor — **alugue um**. Uma VPS de R$ 30/mês por um mês é o melhor investimento deste módulo.
2. Todo 🔴 tem uma armadilha.
3. **Quebre o pipeline de propósito.** Um portão que você nunca viu reprovar é um portão em que você não deveria confiar.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 09
# ═══════════════════════════════════════════════════════════════
import json
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import textwrap
import threading
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("uvicorn[standard]", "uvicorn"), ("pyyaml", "yaml"),
               ("gunicorn", "gunicorn")]:
    _garantir(_p, _m)

import httpx
import uvicorn
import yaml
from fastapi import FastAPI

TEM_GUNICORN = _garantir("gunicorn", "gunicorn")
E_LINUX = platform.system() == "Linux"

print(f"sistema   : {platform.system()}")
print(f"gunicorn  : {'✅ disponível' if TEM_GUNICORN else '⚠️ indisponível (só Unix)'}")


# ═══════════════════════════════════════════════════════════════
#  Servidores de verdade
# ═══════════════════════════════════════════════════════════════

def porta_livre() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class Servico:
    """Sobe uma app ASGI num uvicorn de verdade, em segundo plano."""

    def __init__(self, app, nome: str = "servico", **config):
        self.app, self.nome = app, nome
        self.porta = porta_livre()
        self.url = f"http://127.0.0.1:{self.porta}"
        self.config = config
        self._servidor = self._thread = None

    def iniciar(self, timeout: float = 20.0) -> "Servico":
        cfg = uvicorn.Config(self.app, host="127.0.0.1", port=self.porta,
                             log_level="critical", access_log=False, **self.config)
        self._servidor = uvicorn.Server(cfg)
        self._thread = threading.Thread(target=self._servidor.run, daemon=True)
        self._thread.start()
        limite = time.monotonic() + timeout
        while time.monotonic() < limite:
            if self._servidor.started:
                return self
            time.sleep(0.05)
        raise RuntimeError(f"{self.nome} não subiu")

    def parar(self) -> None:
        if self._servidor is not None:
            self._servidor.should_exit = True
            self._thread.join(timeout=10)

    def __enter__(self):
        return self.iniciar()

    def __exit__(self, *_):
        self.parar()


# ═══════════════════════════════════════════════════════════════
#  Shell e exibição
# ═══════════════════════════════════════════════════════════════

def sh(comando: str, mostrar: bool = True, cwd=None, timeout: int = 120,
       env: dict | None = None) -> subprocess.CompletedProcess:
    p = subprocess.run(comando, shell=True, capture_output=True, text=True,
                       cwd=cwd, timeout=timeout,
                       env=dict(os.environ, **(env or {})))
    if mostrar:
        saida = (p.stdout + p.stderr).rstrip()
        if saida:
            print(saida)
    return p


def referencia(comando: str, esperado: str = "") -> None:
    """Mostra um comando que NÃO roda aqui, com a saída típica.

    🔴 Saída marcada `[referência]` não foi executada. Rode você mesmo —
       é assim que se aprende deploy.
    """
    print(f"$ {comando}")
    if esperado:
        for linha in textwrap.dedent(esperado).strip("\n").splitlines():
            print(f"  {linha}")
    print("  ── [referência] não executado neste ambiente ──")


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def arvore(raiz: Path, prefixo: str = "") -> None:
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".git"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        print(f"{prefixo}{'└── ' if ultimo else '├── '}{item.name}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


print("✅ `Servico`, `sh()`, `referencia()`, `tabela()`, `preparar()` prontos")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Ferramentas das aulas anteriores
# ═══════════════════════════════════════════════════════════════
def ordenar_jobs(jobs: dict) -> list[str]:
    pendentes, ordem = dict(jobs), []
    while pendentes:
        prontos = [
            n for n, j in pendentes.items()
            if all(d in ordem for d in (
                [j["needs"]] if isinstance(j.get("needs"), str)
                else (j.get("needs") or [])))
        ]
        if not prontos:
            raise ValueError(f"🔴 ciclo ou dependência inexistente: {list(pendentes)}")
        for n in sorted(prontos):
            ordem.append(n)
            pendentes.pop(n)
    return ordem


def rodar_workflow(caminho: Path, cwd: Path, segredos: dict | None = None,
                   silencioso: bool = False) -> int:
    """Executa um workflow do GitHub Actions localmente."""
    wf = yaml.safe_load(Path(caminho).read_text(encoding="utf-8"))
    jobs = wf.get("jobs") or {}
    ordem = ordenar_jobs(jobs)
    if not silencioso:
        print(f"workflow: {wf.get('name', '(sem nome)')}")
        print(f"ordem   : {' → '.join(ordem)}\n")

    ambiente = dict(os.environ, **{k: str(v) for k, v in (wf.get("env") or {}).items()})
    for chave, valor in (segredos or {}).items():
        ambiente[chave] = valor

    falharam: list[str] = []
    for nome in ordem:
        job = jobs[nome]
        deps = ([job["needs"]] if isinstance(job.get("needs"), str)
                else (job.get("needs") or []))
        if any(d in falharam for d in deps):
            if not silencioso:
                print(f"⏭️  {nome}: PULADO (dependência falhou)\n")
            falharam.append(nome)
            continue
        if not silencioso:
            print(f"┌─ {nome}")
        env_job = dict(ambiente, **{k: str(v) for k, v in (job.get("env") or {}).items()})
        tudo_ok = True
        for passo in job.get("steps") or []:
            if "uses" in passo:
                if not silencioso:
                    print(f"│  ⏭️  uses: {passo['uses']}")
                continue
            comando = passo.get("run")
            if not comando:
                continue
            rotulo = passo.get("name") or comando.splitlines()[0][:44]
            inicio = time.perf_counter()
            r = subprocess.run(
                comando, shell=True, cwd=cwd, capture_output=True, text=True,
                env=dict(env_job, **{k: str(v) for k, v in (passo.get("env") or {}).items()}))
            ms = (time.perf_counter() - inicio) * 1000
            if not silencioso:
                print(f"│  {'✅' if r.returncode == 0 else '🔴'} {rotulo:<44}{ms:>7.0f}ms")
                if r.returncode != 0:
                    for linha in (r.stdout + r.stderr).strip().splitlines()[-5:]:
                        print(f"│       {linha}")
            if r.returncode != 0 and not passo.get("continue-on-error"):
                tudo_ok = False
                break
        if not silencioso:
            print(f"└─ {'✅' if tudo_ok else '🔴'}\n")
        if not tudo_ok:
            falharam.append(nome)
    return 1 if falharam else 0


FATORES = [
    (1, "Base de código"), (2, "Dependências"), (3, "Configuração"),
    (4, "Serviços de apoio"), (5, "Build/release/run"), (6, "Processos"),
    (7, "Vínculo de porta"), (8, "Concorrência"), (9, "Descartabilidade"),
    (10, "Paridade dev/prod"), (11, "Logs"), (12, "Tarefas admin"),
]

SEGREDO_LITERAL = re.compile(
    r'(SECRET|PASSWORD|SENHA|TOKEN|API_?KEY|CREDENTIAL)\w*\s*=\s*["\'][^"\']{6,}', re.I)

print("✅ `rodar_workflow()` e `ordenar_jobs()` prontos")

---

# 🟢 Aquecimento (1–10)

### 1 · Três ambientes

Liste as diferenças entre dev, homologação e produção do Atlas. Marque o que **pode** mudar e o que **não pode**.

In [ ]:
# 1

### 2 · O artefato único

Explique, num comentário, por que o mesmo commit precisa virar a **mesma imagem** nos três ambientes.

In [ ]:
# 2

### 3 · Auditoria 12-factor

Rode a auditoria da aula 09_01 no seu `projeto_Atlas` e liste os 🔴.

In [ ]:
# 3

### 4 · Uvicorn vs Gunicorn

Suba a mesma app dos dois jeitos e mostre a diferença nos PIDs que atendem.

In [ ]:
# 4

### 5 · 🔴 Estado no processo

Ponha um contador em memória, suba com 3 workers e mostre que ele conta errado.

In [ ]:
# 5

### 6 · Proxy reverso

Escreva um proxy mínimo em Python e mostre os três cabeçalhos chegando na aplicação.

In [ ]:
# 6

### 7 · 🔴 IP forjado

Prove que um cliente pode forjar `X-Forwarded-For`. Depois corrija com `forwarded_allow_ips`.

In [ ]:
# 7

### 8 · Chave SSH

Gere um par ed25519 com senha e mostre o fingerprint. Explique o que a senha protege.

In [ ]:
# 8

### 9 · Symlink atômico

Implemente a troca de release com `os.replace()`. Explique por que `ln -sfn` não basta.

In [ ]:
# 9

### 10 · Primeiro workflow

Escreva um `ci.yml` com dois jobs encadeados e rode com o `rodar_workflow()`.

In [ ]:
# 10

---

# 🟡 Construção (11–26)

### 11 · Número de workers

Calcule pela fórmula tradicional e explique por que ela superestima com `UvicornWorker`.

In [ ]:
# 11

### 12 · 🔴 `os.cpu_count()` mente

Mostre a diferença entre os núcleos do host e o limite do cgroup. Explique o estrago.

In [ ]:
# 12

### 13 · `nginx.conf`

Escreva um com TLS, limite de corpo, limite de taxa e WebSocket. Explique cada `proxy_set_header`.

In [ ]:
# 13

### 14 · 🔴 Sem `X-Forwarded-Proto`

Explique o laço infinito de redirecionamento que aparece sem ele.

In [ ]:
# 14

### 15 · Encerramento gracioso

Demonstre com uma requisição de 5 segundos e um `SIGTERM` no meio.

In [ ]:
# 15

### 16 · `/saude` vs `/pronto`

Implemente as duas e explique qual o orquestrador usa para reiniciar e qual para tirar do balanceador.

In [ ]:
# 16

### 17 · Endurecer o SSH

Liste as cinco linhas do `sshd_config` e explique cada uma. Diga por que manter uma segunda sessão aberta.

In [ ]:
# 17

### 18 · Usuário de deploy

Crie o usuário e o `sudoers.d` mínimo. Justifique o que isso impede se a chave vazar.

In [ ]:
# 18

### 19 · `releases/` + symlink

Implemente `publicar()`, `rollback()` e `limpar()`. Teste a limpeza logo **após** um rollback.

In [ ]:
# 19

### 20 · `rsync`

Monte a lista de exclusões e meça a economia. Compare com o `.dockerignore` do M08.

In [ ]:
# 20

### 21 · `systemd`

Escreva o unit e valide com `systemd-analyze verify`. Explique `TimeoutStopSec` vs `--graceful-timeout`.

In [ ]:
# 21

### 22 · 🔴 Migrações

Classifique cinco migrações suas em compatíveis e incompatíveis. Escreva o plano de três deploys para uma incompatível.

In [ ]:
# 22

### 23 · Portão de segredos

Adicione um job de CI que falhe se houver segredo no código. Commite um e veja barrar.

In [ ]:
# 23

### 24 · 🔴 Vazamento por transformação

Mostre `base64`, substring e `rev` escapando do mascaramento. Explique por quê.

In [ ]:
# 24

### 25 · Log estruturado

Implemente o `FormatadorJSON` com id de correlação. Prove que o campo `senha` nunca sai.

In [ ]:
# 25

### 26 · p95 vs média

Calcule os dois numa amostra com cauda longa e explique por que a média mente.

In [ ]:
# 26

---

# 🔴 Integração (27–40)

### 27 · Pipeline completo

`ci.yml` com qualidade → segurança → testes → auditoria → build. Todos os portões funcionando.

In [ ]:
# 27

### 28 · 🔴 Quebre cada portão

Para **cada** job do seu CI, provoque uma falha e confirme que ele reprova. Um portão nunca testado não é um portão.

In [ ]:
# 28

### 29 · 🔴 Bytecode velho

Reproduza o caso da aula 09_03: altere um arquivo mantendo tamanho e data, e mostre o teste passando errado.

In [ ]:
# 29

### 30 · `cd.yml`

Com `environment` de aprovação, verificação de saúde e rollback em `if: failure()`.

In [ ]:
# 30

### 31 · Script de deploy

Com teste local, envio, migração, symlink, reload, verificação e rollback automático.

In [ ]:
# 31

### 32 · 🔴 Rollback sob pressão

Provoque um deploy ruim e reverta. Cronometre. Se levar mais de 60 segundos, simplifique.

In [ ]:
# 32

### 33 · Deploy sem queda

Faça um deploy enquanto um cliente chama a API 1×/s. Mostre que nenhuma requisição falhou.

In [ ]:
# 33

### 34 · Backup e restauração

Escreva os dois scripts. 🔴 **Teste a restauração** — num banco vazio, do zero.

In [ ]:
# 34

### 35 · Métricas

Implemente `/metricas` no formato Prometheus com os quatro sinais + uma métrica de negócio.

In [ ]:
# 35

### 36 · 🔴 Métrica com N workers

Mostre a métrica em memória contando errado com 3 workers. Proponha a correção.

In [ ]:
# 36

### 37 · Alertas

Classifique dez condições em "acorda", "horário comercial" e "só gráfico". Justifique as que não acordam.

In [ ]:
# 37

### 38 · Runbook

Escreva o que fazer nos três incidentes mais prováveis do Atlas. Escreva para ser lido às 3h da manhã.

In [ ]:
# 38

### 39 · Checklist

Percorra o checklist de produção com o seu Atlas e liste as pendências com prazo.

In [ ]:
# 39

### 40 · 🔴 Simulacro de incidente

Derrube o banco de propósito com a API no ar. Cronometre: quanto tempo até você **descobrir**? E até **resolver**?

In [ ]:
# 40

---

# 🏗️ Projeto — O Atlas no ar, sozinho

## O contexto

> *"Subir versão é um ritual de risco. E a gente descobre que o site caiu quando um cliente liga."*

Ao final:

```bash
git push origin main
# ... e pronto
```

O pipeline testa, audita, publica, verifica e reverte sozinho se der errado. E se algo quebrar em produção, o alerta chega antes do telefone.

## O que entregar

```
projeto_Atlas/
├── .github/workflows/
│   ├── ci.yml                 ← qualidade, segurança, testes, auditoria
│   └── cd.yml                 ← deploy com aprovação e rollback
├── infra/
│   ├── atlas.service          ← systemd
│   ├── nginx.conf             ← proxy reverso + TLS
│   └── docker-compose.prod.yml
├── scripts/
│   ├── deploy.sh              ← com verificação e rollback
│   ├── rollback.sh            ← um comando
│   ├── backup.sh
│   └── restaurar.sh           ← 🔴 e TESTADO
└── docs/
    ├── DEPLOY.md              ← o passo a passo
    └── RUNBOOK.md             ← 🔴 o que fazer quando quebrar
```

## Requisitos obrigatórios

| # | Requisito | Pronto quando |
|---|-----------|---------------|
| 1 | CI com 4+ portões | Cada um reprova quando deve |
| 2 | 🔴 Portão de segredos | Commitar um segredo barra o merge |
| 3 | CD só após CI verde | `if: ...conclusion == 'success'` |
| 4 | Aprovação manual | `environment: producao` |
| 5 | 🔴 Rollback automático | `if: failure()` |
| 6 | Deploy verifica a saúde | Falhou → reverte |
| 7 | `systemd` valida | `systemd-analyze verify` limpo |
| 8 | Nginx com TLS | Renovação testada |
| 9 | 🔴 `forwarded_allow_ips` | IP forjado é ignorado |
| 10 | Log JSON no stdout | Sem dado pessoal |
| 11 | Métrica de negócio | Ao menos uma |
| 12 | 🔴 Restauração testada | Do zero, num banco vazio |
| 13 | `RUNBOOK.md` | Três incidentes documentados |

> 📋 **O roteiro está em `projeto_Atlas/ROTEIRO_M09.md`.**

## 🧪 Bateria de aceitação

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Aponte para o SEU projeto
# ═══════════════════════════════════════════════════════════════
PROJETO = None          # ← troque pelo caminho do seu projeto_Atlas

RESULTADOS = []


def checar(nome: str, condicao: bool, detalhe: str = "") -> bool:
    RESULTADOS.append((nome, bool(condicao)))
    print(f"   {'✅' if condicao else '🔴'} {nome}{('  — ' + detalhe) if detalhe else ''}")
    return bool(condicao)


def placar():
    passou = sum(1 for _, ok in RESULTADOS if ok)
    print(f"\n{'═' * 56}\n  {passou}/{len(RESULTADOS)} verificações passaram")
    if RESULTADOS and passou == len(RESULTADOS):
        print("  🎉 Atlas em produção, sozinho.")
    else:
        for nome, ok in RESULTADOS:
            if not ok:
                print(f"  🔴 pendente: {nome}")
    print("═" * 56)


print("⏸️  defina `PROJETO` acima para rodar a bateria."
      if PROJETO is None else f"▶️  auditando {PROJETO}")

In [ ]:
# ── Bateria 1: os arquivos de infraestrutura ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    fluxos = PROJETO / ".github" / "workflows"
    checar("existe .github/workflows/", fluxos.is_dir())

    arquivos = sorted(fluxos.glob("*.yml")) + sorted(fluxos.glob("*.yaml")) \
        if fluxos.is_dir() else []
    checar("há ao menos 1 workflow", bool(arquivos),
           str([a.name for a in arquivos]))

    for nome in ["scripts/deploy.sh", "scripts/rollback.sh", "docs/DEPLOY.md"]:
        checar(f"existe {nome}", (PROJETO / nome).exists())

    checar("🔴 existe um RUNBOOK", (PROJETO / "docs" / "RUNBOOK.md").exists(),
           "o que fazer quando quebrar, escrito ANTES de quebrar")

    placar()

In [ ]:
# ── Bateria 2: 🔴 o pipeline e seus portões ──
RESULTADOS.clear()

if PROJETO is None or not (PROJETO / ".github" / "workflows").is_dir():
    print("⏸️  defina `PROJETO` e crie os workflows")
else:
    fluxos = PROJETO / ".github" / "workflows"
    todos = list(fluxos.glob("*.yml")) + list(fluxos.glob("*.yaml"))

    for arquivo in todos:
        try:
            wf = yaml.safe_load(arquivo.read_text(encoding="utf-8"))
            jobs = wf.get("jobs") or {}
            checar(f"{arquivo.name}: YAML válido com jobs", bool(jobs),
                   f"{len(jobs)} job(s): {sorted(jobs)}")
            try:
                ordem = ordenar_jobs(jobs)
                checar(f"{arquivo.name}: sem ciclo em needs", True,
                       " → ".join(ordem))
            except ValueError as erro:
                checar(f"{arquivo.name}: sem ciclo em needs", False, str(erro))
        except yaml.YAMLError as erro:
            checar(f"{arquivo.name}: YAML válido", False, str(erro)[:50])

    texto = "\n".join(a.read_text(encoding="utf-8") for a in todos)
    portoes = {
        "testes (pytest)":        "pytest" in texto,
        "lint ou sintaxe":        any(p in texto for p in ("ruff", "flake8", "compileall")),
        "🔴 portão de segredos":  any(p in texto.lower() for p in
                                      ("secret", "senha", "gitleaks", "password")),
        "auditoria de container": any(p in texto for p in
                                      ("auditar", "hadolint", "docker")),
    }
    for nome, existe in portoes.items():
        checar(f"o CI tem {nome}", existe)

    checar("🔴 o CD reverte com if: failure()", "failure()" in texto,
           "sem isto, um deploy ruim fica no ar")
    checar("o CD exige CI verde",
           "conclusion" in texto or "needs" in texto)

    placar()

In [ ]:
# ── Bateria 3: 🔴 nenhum segredo versionado ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    alvos: list[Path] = []
    for pasta, padrao in [(".github", "*.yml"), ("scripts", "*.sh"),
                          ("infra", "*")]:
        if (PROJETO / pasta).is_dir():
            alvos.extend((PROJETO / pasta).rglob(padrao))

    suspeitos = []
    for arquivo in alvos:
        if not arquivo.is_file():
            continue
        for i, linha in enumerate(
                arquivo.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            nua = linha.split("#")[0]
            if SEGREDO_LITERAL.search(nua) and "secrets." not in nua \
               and "${" not in nua:
                suspeitos.append(f"{arquivo.name}:{i}")
    checar("🔴 nenhum segredo literal em workflow/script/infra",
           not suspeitos, str(suspeitos[:3]))

    gitignore = PROJETO / ".gitignore"
    if gitignore.exists():
        conteudo = gitignore.read_text(encoding="utf-8")
        checar(".env ignorado pelo Git", ".env" in conteudo)
        checar("chaves privadas ignoradas",
               any(p in conteudo for p in ("*.pem", "*.key", "id_ed25519")))

    checar("existe .env.example", (PROJETO / ".env.example").exists())

    placar()

In [ ]:
# ── Bateria 4: 🔴 o pipeline REPROVA quando deve ──
#
# 🎯 A bateria mais importante das quatro.
#
#    As anteriores confirmam que o pipeline EXISTE. Esta confirma que
#    ele FUNCIONA — e é a única que a maioria das pessoas não faz.
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    ci = next((p for p in (PROJETO / ".github" / "workflows").glob("ci*.y*ml")), None)
    if ci is None:
        print("⏸️  não achei um ci.yml")
    else:
        print("   ── 1. o pipeline passa no código atual? ──")
        limpo = rodar_workflow(ci, cwd=PROJETO, silencioso=True)
        checar("CI verde no código atual", limpo == 0,
               "se já está vermelho, conserte antes de continuar")

        if limpo == 0:
            print("\n   ── 2. e reprova com um segredo plantado? ──")
            armadilha = PROJETO / "src" / "_armadilha_temporaria.py"
            armadilha.parent.mkdir(parents=True, exist_ok=True)
            armadilha.write_text(
                'ATLAS_SECRET_KEY = "chave-de-producao-plantada-123456"\n',
                encoding="utf-8")
            try:
                com_segredo = rodar_workflow(ci, cwd=PROJETO, silencioso=True)
                checar("🔴 CI reprova código com segredo", com_segredo != 0,
                       "se ficou verde, o portão de segredos NÃO funciona")
            finally:
                armadilha.unlink(missing_ok=True)
                for cache in PROJETO.rglob("__pycache__"):
                    shutil.rmtree(cache, ignore_errors=True)

        placar()

> 🎯 **A bateria 4 é a que separa um pipeline de um enfeite.**
>
> As três primeiras conferem que os arquivos existem e estão bem formados. A quarta planta um segredo e verifica que o CI **reprova**.
>
> 🔴 **E a diferença entre as duas não é teórica.** Ao testar estas baterias, desliguei o portão de segredos trocando o `grep` por um `echo` — e o resultado foi:
>
> ```
> Bateria 2:  ✅ o CI tem 🔴 portão de segredos
> Bateria 4:  🔴 CI reprova código com segredo   ← reprovou
> ```
>
> A bateria 2 procurou a palavra "secret" no arquivo e a encontrou. O portão **existia** e não **funcionava**. Só executá-lo revela isso.
>
> 💭 Guarde o padrão: *"o arquivo menciona X"* é uma evidência muito fraca de que X está sendo feito.
>
> 💭 É a mesma ideia que atravessa o manual inteiro, aplicada agora ao seu próprio ferramental:
>
> | Módulo | A mesma lição |
> |--------|---------------|
> | M07 | Um teste que nunca falhou não prova nada |
> | M08 | Um verificador que nunca reprova ninguém não verifica |
> | M09 | Um portão de CI não testado é um portão aberto |
>
> **Antes de confiar em qualquer verificação, quebre o código de propósito e veja se ela reclama.** Se não reclamar, você tem uma falsa sensação de segurança — que é pior do que não ter verificação nenhuma, porque você para de olhar.

---

## 🎓 Autoavaliação

| # | Consigo… | ✅ |
|---|----------|---|
| 1 | Explicar por que homologação existe | |
| 2 | Justificar o artefato único nos três ambientes | |
| 3 | Citar os 12 fatores e os 4 que mais doem | |
| 4 | Explicar a diferença entre uvicorn e gunicorn | |
| 5 | 🔴 Listar o que quebra com vários workers | |
| 6 | Explicar por que `os.cpu_count()` mente no container | |
| 7 | Dizer o que o proxy reverso resolve, inclusive cliente lento | |
| 8 | 🔴 Explicar por que `X-Forwarded-For` é forjável | |
| 9 | Dizer o que quebra sem `X-Forwarded-Proto` | |
| 10 | Explicar onde o TLS termina | |
| 11 | Descrever o encerramento gracioso passo a passo | |
| 12 | Explicar por que tirar do balanceador antes de encerrar | |
| 13 | Listar cinco linhas de endurecimento do SSH | |
| 14 | Explicar o padrão `releases/` + symlink | |
| 15 | Dizer por que `ln -sfn` não é atômico | |
| 16 | Explicar por que o `.env` mora em `compartilhado/` | |
| 17 | Classificar migrações em compatíveis e incompatíveis | |
| 18 | Descrever o plano de três deploys para renomear coluna | |
| 19 | Explicar por que migração não roda no start-up | |
| 20 | Fazer rollback em menos de 60 segundos | |
| 21 | Explicar por que rollback de banco é outro problema | |
| 22 | Ordenar as etapas do CI por custo crescente | |
| 23 | 🔴 Explicar por que o mascaramento de segredo falha | |
| 24 | Dizer o que fazer quando um segredo vaza | |
| 25 | Justificar (ou não) deploy automático em produção | |
| 26 | Explicar por que a média mente e o p95 não | |
| 27 | Citar três métricas de negócio da Aurora | |
| 28 | Explicar a fadiga de alerta e como evitá-la | |
| 29 | 🔴 Explicar por que o CI parte de um clone limpo | |
| 30 | Provar que os meus portões de CI reprovam quando devem | |

**Menos de 24?** Volte às aulas antes do Módulo 10.

---

### ➡️ Próximo módulo

**Módulo 10 — Engenharia de Dados.** *"Decidimos com dados de três semanas atrás."*

O Atlas está no ar, seguro e monitorado. Agora a diretoria quer responder perguntas que o banco transacional não responde — e para isso entram Pandas, Polars, Parquet, ETL e orquestração.